# Primera actividad: las direcciones de los neutrinos

Para resolver el misterio del origen de los rayos cósmicos, necesitamos poder *reconstruir* las direcciones de los neutrinos a partir de nuestras observaciones.

En esta actividad, ¡van a intentar hacer esta reconstrucción!

In [ ]:
# Ejecuten esta celda para descargar los dados
# y prepar las funciones que necesitamos.

from google.colab import output
output.enable_custom_widget_manager()

!rm -rf sample_data
!mkdir -p sim_moonshadow

# download tracks.parquet
!gdown 1nqffw6xHLdX2oO8d-5xjExYZryo7rp3x
!mv tracks.parquet sim_moonshadow/tracks.parquet

# download cascades.parquet
!gdown 1yo3jD0a9xB2FfIXJJuMRc71IMe6nztvq
!mv cascades.parquet sim_moonshadow/cascades.parquet

# download pre-prepared analysis code
!rm -rf IceCube_MasterClass_at_Harvard
!git clone "https://github.com/Harvard-Neutrino/IceCube_MasterClass_at_Harvard.git";

# load analysis code
import sys
sys.path.insert(0, "./IceCube_MasterClass_at_Harvard/")
from src.plot_event import *
from src.reco_game import reco_game
from src.event_reader import load_sim_events
from src.direction_utils import *
import math
def angle_deg(u, v):
    # dot product
    d = u[0]*v[0] + u[1]*v[1] + u[2]*v[2]
    # magnitudes
    nu = math.hypot(u[0], u[1], u[2])
    nv = math.hypot(v[0], v[1], v[2])
    # cosine of angle, clamped
    cos_t = max(min(d/(nu*nv), 1.0), -1.0)
    return math.degrees(math.acos(cos_t))

# load the track event set:
tracks = load_sim_events("sim_moonshadow/tracks.parquet")

In [ ]:
# ¿qué contiene el objeto `tracks`?
tracks

In [ ]:
# Un evento es simplemente un grupo de “hits”: momentos en los que
# la luz llegó a diferentes sensores. Estos hits se agrupan porque
# ocurrieron dentro de la misma ventana de tiempo.

# Podemos investigar el contenido del primer evento:
evt_num = 1               # ¡Cámbiame para ver otro evento!
evt = tracks[evt_num]
evt

In [ ]:
# Una lista de hits no nos dice mucho por sí sola.
# Podemos *visualizar* un evento dentro del detector haciendo una gráfica.
fig = display_evt( evt )
fig.show()

In [ ]:
# Ahora vamos a empezar a reconstruir las direcciones de los muones.
# Preparamos un juego para calentar motores.

# ¿Cuál es el mejor resultado que puedes lograr?
# Cuando termines, haz clic en "return" para volver y escoger otro evento.
reco_game(tracks)

In [ ]:
# También podemos usar un algoritmo para ayudarnos.
# Nuestro algoritmo calcula la distancia perpendicular promedio
# entre los detectores y el rayo de nuestra dirección estimada.

"""
Esta función calcula la distancia perpendicular entre un solo punto y un rayo,
usando un punto de referencia en el rayo.
"""
def calc_perpendicular_distance( hit_pt, dir_vec, pivot_pt ):
    dist_vec = hit_pt - pivot_pt
    return np.sqrt( np.dot( dist_vec, dist_vec ) -  np.dot( dist_vec, dir_vec )**2 )

"""
Esta función calcula un punto de referencia a partir de un conjunto de hits.
"""
def calc_center_of_gravity( hits ):
    return hits.mean(axis=0)

"""
Dado un conjunto de hits y un par de ángulos estimados, calcula el promedio
de todas las distancias perpendiculares.
"""
def calc_mean_perpendicular_distance( dir_angles, hit_pts, pivot_pt ):

    dir_vec = get_direction_vector_from_angles( dir_angles[0], dir_angles[1] )
    return np.mean([ calc_perpendicular_distance(hit, dir_vec, pivot_pt) for hit in hit_pts ])


In [ ]:
# Ahora vamos a jogar de nuevo, usando nuestro algoritmo para ayudarnos.
reco_game( tracks, calc_dist=True )

In [ ]:
# Por último, podemos usar la computadora para *minimizar*
# la distancia perpendicular promedio.

# Importamos la función que hace la minimización:
from scipy.optimize import minimize

# Definimos la función que le vamos a pasar a `minimize`:
def function_to_minimize( dir_angles ):
    return calc_mean_perpendicular_distance(
        dir_angles,
        evt.hits_xyz,
        calc_center_of_gravity(evt.hits_xyz)
    )

def make_initial_guess_for_angles( evt ):

    j0, j1 = np.argmin( evt.hits_t ), np.argmax( evt.hits_t )
    if j0 == j1:
        initial_guess_azi = np.deg2rad( -10 )
        initial_guess_zen = np.deg2rad( 180 )
        return np.array( [initial_guess_azi, initial_guess_zen] )

    initial_guess = get_direction_angles_from_vector(
        normalize( evt.hits_xyz[j1, :] - evt.hits_xyz[j0, :] )
    )
    return initial_guess

In [ ]:
# Ahora podemos ver qué tan bien la computadora reconstruye la dirección correcta.

# ¡Cámbiame para ver otro evento!
evt_num = 1
evt = tracks[evt_num]

# Hacemos la minimización:
minimization_output = minimize(
    function_to_minimize,
    make_initial_guess_for_angles( evt ),
)

best_guess_azi, best_guess_zen = minimization_output.x
smallest_distance = minimization_output.fun

# Imprimimos los resultados:
print("Los ángulos con la distancia perpendicular más pequeña son:")
print(f"azi = {np.rad2deg(best_guess_azi):.2f} grados")
print(f"zen = {np.rad2deg(best_guess_zen):.2f} grados")
print("Para estos ángulos, la distancia perpendicular promedio es:\n")
print(f"\t {smallest_distance:.2f} metros")

# Visualizamos la dirección reconstruida junto con la dirección correcta:
dir_vec = get_direction_vector_from_angles( best_guess_azi, best_guess_zen )
pivot_pt = calc_center_of_gravity(evt.hits_xyz)
fig = display_evt( evt )
fig.add_traces( plot_direction( dir_vec, pivot_pt, color="hotpink" ) )

# Ahora calculamos la dirección verdadera del muón:
true_dir_vec = get_direction_vector_from_angles(
    evt.true_muon_azimuth,
    evt.true_muon_zenith
)

print(
    "La estimación obtenida al minimizar la distancia perpendicular se desvió "
    + str(round(angle_deg(dir_vec, true_dir_vec), 2))
    + "° de la dirección verdadera.\n"
)

# Visualizamos la dirección reconstruida junto con la dirección verdadera:
fig.add_traces(plot_direction(true_dir_vec, pivot_pt, color="black"))
fig.show()

In [ ]:
# Fin!